In [ ]:
%pip install statsbombpy duckdb pandas numpy matplotlib mplsoccer scikit-learn scipy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
from statsbombpy import sb

In [ ]:
# 1. Afficher les compétitions gratuites disponibles
competitions = sb.competitions()
print("Exemples de compétitions disponibles :")
print(competitions[['competition_id', 'season_id', 'competition_name', 'season_name']].drop_duplicates().tail(10))

In [ ]:
# 2. Charger les matchs de l'UEFA Euro 2024 (competition_id=55, season_id=282)
COMP_ID = 55
SEASON_ID = 282

In [ ]:
matches = sb.matches(competition_id=COMP_ID, season_id=SEASON_ID)
print(f"\nNombre de matchs chargés : {len(matches)}")
matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']].head()

In [ ]:
# Extraction des événements pour tous les matchs de la compétition
all_events = []

for match_id in matches['match_id']:
    df_match = sb.events(match_id=match_id)
    all_events.append(df_match)

events = pd.concat(all_events, ignore_index=True)

In [ ]:
# Extraction des coordonnées (x, y) depuis la colonne 'location' (qui est une liste [x, y])
events['x'] = events['location'].apply(lambda loc: loc[0] if isinstance(loc, list) and len(loc) >= 2 else np.nan)
events['y'] = events['location'].apply(lambda loc: loc[1] if isinstance(loc, list) and len(loc) >= 2 else np.nan)

In [ ]:
# Extraction des coordonnées de fin de passe/tir ('pass_end_location' ou 'shot_end_location')
events['pass_end_x'] = events['pass_end_location'].apply(lambda loc: loc[0] if isinstance(loc, list) and len(loc) >= 2 else np.nan)
events['pass_end_y'] = events['pass_end_location'].apply(lambda loc: loc[1] if isinstance(loc, list) and len(loc) >= 2 else np.nan)

print(f"Total d'événements extraits : {len(events):,}")
print("Types d'événements principaux :\n", events['type'].value_counts().head(8))

In [ ]:
# Durée max de chaque match (en minutes)
match_durations = events.groupby('match_id')['minute'].max().reset_index().rename(columns={'minute': 'match_max_min'})

In [ ]:
# Récupérer les titulaires (Starting XI) et les entrants (Substitution)
starting_xi = events[events['type'] == 'Starting XI'][['match_id', 'team', 'tactics']]

starters_list = []
for _, row in starting_xi.iterrows():
    lineup = row['tactics']['lineup']
    for player_info in lineup:
        starters_list.append({
            'match_id': row['match_id'],
            'team': row['team'],
            'player_id': player_info['player']['id'],
            'player': player_info['player']['name'],
            'position': player_info['position']['name'],
            'min_in': 0
        })

df_starters = pd.DataFrame(starters_list)

In [ ]:
# Joueurs entrés en cours de match
subs_in = events[events['type'] == 'Substitution'][['match_id', 'team', 'substitution_replacement_id', 'substitution_replacement', 'minute']].copy()
subs_in.columns = ['match_id', 'team', 'player_id', 'player', 'min_in']
subs_in['position'] = 'Substitute'

In [ ]:
# Combinaison de tous les joueurs ayant participé à un match
appearances = pd.concat([df_starters, subs_in], ignore_index=True)

In [ ]:
# Joueurs sortis en cours de match
subs_out = events[events['type'] == 'Substitution'][['match_id', 'player_id', 'minute']].rename(columns={'minute': 'min_out'})

In [ ]:
# Fusion pour calculer les minutes jouées par match
appearances = appearances.merge(subs_out, on=['match_id', 'player_id'], how='left')
appearances = appearances.merge(match_durations, on='match_id', how='left')
appearances['min_out'] = appearances['min_out'].fillna(appearances['match_max_min'])
appearances['minutes_played'] = (appearances['min_out'] - appearances['min_in']).clip(lower=1)

In [ ]:
# Agrégation globale par joueur (Dimension Joueurs)
dim_players = (
    appearances.groupby(['player_id', 'player', 'team'])
    .agg(
        matches_played=('match_id', 'nunique'),
        total_minutes=('minutes_played', 'sum'),
        main_position=('position', lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown')
    )
    .reset_index()
)

print(f"Nombre total de joueurs : {len(dim_players)}")
dim_players.sort_values(by='total_minutes', ascending=False).head(10)

In [ ]:
import os
os.makedirs('data', exist_ok=True)

In [ ]:
# Sélection des colonnes utiles pour la table de faits (fact_events)
cols_events = [
    'id', 'match_id', 'index', 'period', 'timestamp', 'minute', 'second',
    'type', 'team', 'player_id', 'player', 'position',
    'x', 'y', 'pass_end_x', 'pass_end_y',
    'pass_outcome', 'pass_goal_assist', 'pass_shot_assist', 'pass_length',
    'shot_statsbomb_xg', 'shot_outcome', 'shot_type',
    'duel_outcome', 'interception_outcome', 'under_pressure'
    ]

In [ ]:
# Ne garder que les colonnes existantes dans le dataframe
cols_events = [c for c in cols_events if c in events.columns]
fact_events = events[cols_events].copy()

In [ ]:
# Table dimension matchs
dim_matches = matches[['match_id', 'match_date', 'competition', 'season', 'home_team', 'away_team', 'home_score', 'away_score', 'competition_stage']].copy()

In [ ]:
# Connexion à DuckDB et création des tables
con = duckdb.connect('data/football_analytics.duckdb')
con.execute("CREATE OR REPLACE TABLE dim_matches AS SELECT * FROM dim_matches")
con.execute("CREATE OR REPLACE TABLE dim_players AS SELECT * FROM dim_players")
con.execute("CREATE OR REPLACE TABLE fact_events AS SELECT * FROM fact_events")

print("Tables créées dans DuckDB :", con.execute("SHOW TABLES").fetchall())
con.close()


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch, VerticalPitch, PyPizza


In [ ]:
# Connexion à la base créée à l'étape 1
con = duckdb.connect('data/football_analytics.duckdb')

query_kpis = """
CREATE OR REPLACE TABLE mart_player_stats_per90 AS
WITH shot_stats AS (
    SELECT
        player_id,
        COUNT(*) AS total_shots,
        SUM(CASE WHEN shot_outcome = 'Goal' THEN 1 ELSE 0 END) AS goals,
        ROUND(SUM(COALESCE(shot_statsbomb_xg, 0)), 2) AS total_xg
    FROM fact_events
    WHERE type = 'Shot' AND shot_type != 'Penalty'
    GROUP BY player_id
),
pass_stats AS (
    SELECT
        player_id,
        COUNT(*) AS total_passes,
        SUM(CASE WHEN pass_outcome IS NULL THEN 1 ELSE 0 END) AS completed_passes,
        SUM(CASE WHEN pass_shot_assist = TRUE THEN 1 ELSE 0 END) AS key_passes,
        SUM(CASE WHEN pass_goal_assist = TRUE THEN 1 ELSE 0 END) AS assists,
        -- Passes progressives : avancée d'au moins 15 mètres vers le but adverse (x=120)
        SUM(CASE WHEN pass_outcome IS NULL AND (pass_end_x - x) >= 15 AND x >= 40 THEN 1 ELSE 0 END) AS progressive_passes,
        -- Entrées dans le dernier tiers (x >= 80)
        SUM(CASE WHEN pass_outcome IS NULL AND x < 80 AND pass_end_x >= 80 THEN 1 ELSE 0 END) AS final_third_entries
    FROM fact_events
    WHERE type = 'Pass'
    GROUP BY player_id
),
defensive_stats AS (
    SELECT
        player_id,
        SUM(CASE WHEN type = 'Pressure' THEN 1 ELSE 0 END) AS pressures,
        SUM(CASE WHEN type = 'Ball Recovery' THEN 1 ELSE 0 END) AS ball_recoveries,
        SUM(CASE WHEN type = 'Interception' THEN 1 ELSE 0 END) AS interceptions
    FROM fact_events
    GROUP BY player_id
)
SELECT
    p.player_id,
    p.player,
    p.team,
    p.main_position,
    p.matches_played,
    p.total_minutes,
    -- Métriques brutes
    COALESCE(s.goals, 0) AS goals,
    COALESCE(s.total_xg, 0) AS total_xg,
    COALESCE(pa.assists, 0) AS assists,
    -- Métriques normalisées par 90 minutes
    ROUND(COALESCE(s.goals, 0) * 90.0 / p.total_minutes, 2) AS goals_p90,
    ROUND(COALESCE(s.total_xg, 0) * 90.0 / p.total_minutes, 2) AS xg_p90,
    ROUND((COALESCE(s.goals, 0) - COALESCE(s.total_xg, 0)) * 90.0 / p.total_minutes, 2) AS goals_minus_xg_p90,
    ROUND(COALESCE(s.total_shots, 0) * 90.0 / p.total_minutes, 2) AS shots_p90,
    ROUND(COALESCE(pa.key_passes, 0) * 90.0 / p.total_minutes, 2) AS key_passes_p90,
    ROUND(COALESCE(pa.progressive_passes, 0) * 90.0 / p.total_minutes, 2) AS prog_passes_p90,
    ROUND(COALESCE(pa.final_third_entries, 0) * 90.0 / p.total_minutes, 2) AS final_third_entries_p90,
    ROUND(COALESCE(pa.completed_passes, 0) * 100.0 / NULLIF(pa.total_passes, 0), 1) AS pass_accuracy_pct,
    ROUND(COALESCE(d.pressures, 0) * 90.0 / p.total_minutes, 2) AS pressures_p90,
    ROUND((COALESCE(d.ball_recoveries, 0) + COALESCE(d.interceptions, 0)) * 90.0 / p.total_minutes, 2) AS recoveries_interceptions_p90
FROM dim_players p
LEFT JOIN shot_stats s ON p.player_id = s.player_id
LEFT JOIN pass_stats pa ON p.player_id = pa.player_id
LEFT JOIN defensive_stats d ON p.player_id = d.player_id
WHERE p.total_minutes >= 270
ORDER BY xg_p90 DESC;
"""

con.execute(query_kpis)
df_stats = con.execute("SELECT * FROM mart_player_stats_per90").df()
print(f"Joueurs qualifiés (>= 270 min) : {len(df_stats)}")
df_stats[['player', 'team', 'main_position', 'total_minutes', 'goals', 'xg_p90', 'key_passes_p90', 'prog_passes_p90']].head(10)

In [ ]:
# Choisir une équipe présente dans le tournoi (ex: 'Spain', 'France', 'Germany', 'England')
TEAM_NAME = 'Spain'

shots_df = con.execute(f"""
    SELECT player, x, y, shot_statsbomb_xg, shot_outcome
    FROM fact_events
    WHERE type = 'Shot' AND team = '{TEAM_NAME}' AND shot_type != 'Penalty'
""").df()

goals = shots_df[shots_df['shot_outcome'] == 'Goal']
non_goals = shots_df[shots_df['shot_outcome'] != 'Goal']

pitch = VerticalPitch(pitch_type='statsbomb', half=True, pitch_color='#1e212b', line_color='#c7d5cc')
fig, ax = pitch.draw(figsize=(10, 7))
fig.set_facecolor('#1e212b')

# Tirs non convertis
pitch.scatter(
    non_goals.x, non_goals.y,
    s=(non_goals.shot_statsbomb_xg * 600) + 40,
    c='#4ea8de', alpha=0.5, edgecolors='white', ax=ax, label='Tir non converti'
)

# Buts marqués
pitch.scatter(
    goals.x, goals.y,
    s=(goals.shot_statsbomb_xg * 600) + 80,
    c='#ff4d6d', marker='*', edgecolors='white', linewidth=1.2, ax=ax, label='But'
)

ax.set_title(
    f"Shot Map - {TEAM_NAME} (Euro 2024, hors penalties)\nTotal Tirs : {len(shots_df)} | Buts : {len(goals)} | xG Cumulé : {shots_df['shot_statsbomb_xg'].sum():.2f}",
    color='white', fontsize=14, fontweight='bold', pad=15
)
ax.legend(facecolor='#1e212b', edgecolor='white', labelcolor='white', loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import percentileofscore

# Sélection des variables numériques par 90 min pour le profilage
feature_cols = [
    'goals_p90', 'xg_p90', 'shots_p90', 'key_passes_p90',
    'prog_passes_p90', 'final_third_entries_p90', 'pass_accuracy_pct',
    'pressures_p90', 'recoveries_interceptions_p90'
]

# Standardisation des métriques
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_stats[feature_cols].fillna(0))

# Matrice de similarité cosinus
sim_matrix = cosine_similarity(X_scaled)

def find_similar_players(player_query, top_n=5):
    matches = df_stats[df_stats['player'].str.contains(player_query, case=False, na=False)]
    if matches.empty:
        print(f"Aucun joueur trouvé pour '{player_query}'.")
        return None

    idx = matches.index[0]
    target_name = df_stats.loc[idx, 'player']

    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    results = df_stats.iloc[[i[0] for i in scores]][['player', 'team', 'main_position', 'total_minutes', 'xg_p90', 'key_passes_p90', 'prog_passes_p90']].copy()
    results['similarity_score'] = [round(i[1] * 100, 1) for i in scores]

    print(f"🔍 Top {top_n} profils similaires à : {target_name} ({df_stats.loc[idx, 'team']})")
    return target_name, results

# Tester la fonction (tu peux changer le nom par 'Rodri', 'Bellingham', 'Yamal', 'Griezmann', etc.)
target_player, similar_df = find_similar_players("Dani Olmo", top_n=5)
display(similar_df)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import percentileofscore

# Afficher quelques joueurs disponibles (ex: Espagne et France) pour voir le format exact des noms StatsBomb
print("Exemples de joueurs disponibles dans df_stats :")
print(df_stats[df_stats['team'].isin(['Spain', 'France'])][['player', 'team', 'total_minutes']].to_string(index=False))
print("-" * 60)

# Sélection des variables numériques par 90 min pour le profilage
feature_cols = [
    'goals_p90', 'xg_p90', 'shots_p90', 'key_passes_p90',
    'prog_passes_p90', 'final_third_entries_p90', 'pass_accuracy_pct',
    'pressures_p90', 'recoveries_interceptions_p90'
]

# Standardisation des métriques
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_stats[feature_cols].fillna(0))

# Matrice de similarité cosinus
sim_matrix = cosine_similarity(X_scaled)

def find_similar_players(player_query, top_n=5):
    matches = df_stats[df_stats['player'].str.contains(player_query, case=False, na=False)]
    if matches.empty:
        print(f"⚠️ Aucun joueur trouvé pour '{player_query}'. Choisis un nom parmi la liste ci-dessus.")
        return None, None

    idx = matches.index[0]
    target_name = df_stats.loc[idx, 'player']

    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    results = df_stats.iloc[[i[0] for i in scores]][['player', 'team', 'main_position', 'total_minutes', 'xg_p90', 'key_passes_p90', 'prog_passes_p90']].copy()
    results['similarity_score'] = [round(i[1] * 100, 1) for i in scores]

    print(f"🔍 Top {top_n} profils similaires à : {target_name} ({df_stats.loc[idx, 'team']})")
    return target_name, results

# Tester avec "Olmo" (qui correspondra à "Daniel Olmo Carvajal")
target_player, similar_df = find_similar_players("Olmo", top_n=5)
if similar_df is not None:
    display(similar_df)

In [ ]:
# Comparaison visuelle entre le joueur cible et le 1er joueur similaire trouvé
best_match_player = similar_df.iloc[0]['player']

params_labels = [
    "Buts /90", "xG /90", "Tirs /90", "Passes Clés /90",
    "Passes Prog. /90", "Entrées 3e tiers", "% Passes Réussies",
    "Pressings /90", "Récup. + Interc."
]

# Calcul des percentiles sur l'ensemble des joueurs qualifiés
p1_row = df_stats[df_stats['player'] == target_player].iloc[0]
p2_row = df_stats[df_stats['player'] == best_match_player].iloc[0]

p1_values = [int(percentileofscore(df_stats[col], p1_row[col])) for col in feature_cols]
p2_values = [int(percentileofscore(df_stats[col], p2_row[col])) for col in feature_cols]

baker = PyPizza(
    params=params_labels,
    background_color="#1e212b",
    straight_line_color="#000000",
    straight_line_lw=1,
    last_circle_lw=1,
    other_circle_lw=0,
    inner_circle_size=15
)

fig, ax = baker.make_pizza(
    p1_values,
    compare_values=p2_values,
    figsize=(8, 8),
    kwargs_slices=dict(facecolor="#1f77b4", edgecolor="#000000", zorder=2, linewidth=1),
    kwargs_compare=dict(facecolor="#ff7f0e", edgecolor="#000000", zorder=2, linewidth=1),
    kwargs_params=dict(color="white", fontsize=10, va="center"),
    kwargs_values=dict(color="white", fontsize=10, bbox=dict(edgecolor="#000000", facecolor="#1f77b4", boxstyle="round,pad=0.2")),
    kwargs_compare_values=dict(color="white", fontsize=10, bbox=dict(edgecolor="#000000", facecolor="#ff7f0e", boxstyle="round,pad=0.2"))
)

fig.text(0.515, 0.97, f"{target_player} vs {best_match_player} (Percentiles Euro 2024)", size=14, ha="center", color="white", fontweight="bold")
fig.text(0.515, 0.935, f"Bleu : {target_player}  |  Orange : {best_match_player} (Similarité : {similar_df.iloc[0]['similarity_score']}%)", size=11, ha="center", color="#cccccc")
plt.show()

In [ ]:
import os
import shutil
import duckdb
from google.colab import files

# 1. Créer les dossiers locaux sur Colab
os.makedirs('football_exports/images', exist_ok=True)
os.makedirs('football_exports/data', exist_ok=True)

# 2. Sauvegarder le Pizza Plot actuel (si la figure est encore active)
if 'fig' in locals():
    fig.savefig('football_exports/images/scouting_pizza_plot.png', dpi=300, bbox_inches='tight', facecolor='#1e212b')

# 3. Exporter les tables propres en CSV pour Power BI
con = duckdb.connect('data/football_analytics.duckdb')

con.execute("COPY mart_player_stats_per90 TO 'football_exports/data/player_stats_per90.csv' (HEADER, DELIMITER ',')")
con.execute("COPY dim_matches TO 'football_exports/data/dim_matches.csv' (HEADER, DELIMITER ',')")

con.execute("""
    COPY (
        SELECT match_id, team, player, x, y, shot_statsbomb_xg AS xg, shot_outcome
        FROM fact_events
        WHERE type = 'Shot' AND shot_type != 'Penalty'
    ) TO 'football_exports/data/shots_data.csv' (HEADER, DELIMITER ',')
""")

con.close()

# 4. Compresser le dossier en ZIP et lancer le téléchargement automatique sur ton PC
shutil.make_archive('football_scouting_exports', 'zip', 'football_exports')
print("✅ Archive créée ! Le téléchargement va démarrer automatiquement...")
files.download('football_scouting_exports.zip')